# 01 — Exploration des données (EDA)

Analyse exploratoire du dataset d'entraînement `historique_labels_poc.xlsx` :
distribution des thèmes, longueur des verbatims, corrélation satisfaction/sentiment,
classes rares et taux de signaux. **Données simulées — aucune PII réelle.**


In [1]:
import sys, os
# Se placer à la racine du projet (le notebook est dans notebooks/).
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("Racine projet :", ROOT)


Racine projet : /Users/jonathan.dupau/Library/CloudStorage/OneDrive2-EXALT/CULTURA IA VERBATIM/cultura-verbatim-classifier


In [3]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme()
except Exception:
    sns=None
from src.utils import load_config, resolve_path, Taxonomy
from src.preprocessing import load_historique
cfg = load_config()
tax = Taxonomy.from_json(resolve_path(cfg, cfg['paths']['taxonomy']))
df = load_historique(resolve_path(cfg, cfg['paths']['historique']), cfg)
print('Verbatims :', len(df), '| niv1 :', tax.n_niv1, '| niv2 :', tax.n_niv2)
df.head(3)


Verbatims : 7000 | niv1 : 20 | niv2 : 67


,source,date,verbatim_original,satisfaction_score,nb_themes,theme1_niv1,theme1_niv2,theme1_sentiment,theme2_niv1,theme2_niv2,theme2_sentiment,signal_rupture_client,signal_churn,signal_insatisfaction_forte,__source__,__text_raw__,__satisfaction__
0,MDTC,10/02/2025,Mes points en magasin et en ligne ne sont pas ...,2,1,Expérience omnicanale,Programme fidélité non unifié,Négatif,NaN,NaN,NaN,False,True,True,MDTC,Mes points en magasin et en ligne ne sont pas ...,2
1,MDTC,17/04/2024,Produit exactement conforme à la description. ...,9,1,Produit,Non conforme à la description,Positif,NaN,NaN,NaN,False,False,False,MDTC,Produit exactement conforme à la description. ...,9
2,MDTC,04/07/2024,Points crédités immédiatement après ma commande.,10,1,Programme de fidélité,Points non crédités,Positif,NaN,NaN,NaN,False,False,False,MDTC,Points crédités immédiatement après ma commande.,10


## Distribution des grandes thématiques (niv.1)


In [ ]:
vc = df['theme1_niv1'].value_counts()
ax = vc.plot(kind='barh', figsize=(8,7)); ax.invert_yaxis()
ax.set_title('Répartition des thèmes niv.1 (theme1)'); plt.tight_layout(); plt.show()
vc


## Sentiment, nombre de thèmes et score de satisfaction


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(14,4))
df['theme1_sentiment'].value_counts().plot(kind='bar', ax=axes[0], title='Sentiment')
df['nb_themes'].value_counts().plot(kind='bar', ax=axes[1], title='nb_themes')
df['satisfaction_score'].plot(kind='hist', bins=10, ax=axes[2], title='Satisfaction (1-10)')
plt.tight_layout(); plt.show()


## Corrélation score de satisfaction ↔ sentiment


In [ ]:
ct = pd.crosstab(df['satisfaction_score'], df['theme1_sentiment'], normalize='index')
ct.plot(kind='bar', stacked=True, figsize=(10,4), title='Sentiment selon le score de satisfaction')
plt.tight_layout(); plt.show()
ct.round(2)


## Longueur des verbatims (en mots)


In [ ]:
df['n_mots'] = df['verbatim_original'].astype(str).str.split().apply(len)
df['n_mots'].plot(kind='hist', bins=40, figsize=(9,3), title='Longueur des verbatims (mots)')
plt.tight_layout(); plt.show()
df['n_mots'].describe()


## Classes rares (sous-thèmes niv.2) et taux de signaux


In [ ]:
from collections import Counter
cnt = Counter()
for _,r in df.iterrows():
    cnt[r['theme1_niv2']] += 1
    if pd.notna(r['theme2_niv2']): cnt[r['theme2_niv2']] += 1
rares = {k:v for k,v in cnt.items() if v < 50}
print('Sous-thèmes < 50 exemples :', rares)
for s in ['signal_rupture_client','signal_churn','signal_insatisfaction_forte']:
    print(f'{s:32s} taux = {df[s].mean()*100:5.2f}%  (n+={int(df[s].sum())})')


**Observations clés** : thèmes globalement équilibrés ; sentiment quasi-équilibré ; multi-thèmes très rares (~0,4 %) ; `signal_rupture_client` extrêmement minoritaire (~0,4 %) → justifie le pondération de classe et la priorité au rappel.
